In [2]:
from pathlib import Path
import pandas as pd
import networkx as nx
from networkx.algorithms.community import louvain_communities


def louvain(file_path: str) -> str:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Input file not found: {file_path}")

    df = pd.read_csv(file_path)

    required_cols = ["source", "destination", "timestamp"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}. Required: {required_cols}")

    df["source"] = df["source"].astype(str)
    df["destination"] = df["destination"].astype(str)

    u = df["source"]
    v = df["destination"]
    a = u.where(u <= v, v)
    b = v.where(u <= v, u)

    edge_w = (
        pd.DataFrame({"u": a, "v": b})
        .groupby(["u", "v"], as_index=False)
        .size()
        .rename(columns={"size": "weight"})
    )

    G = nx.Graph()
    nodes = pd.Index(df["source"]).append(pd.Index(df["destination"])).unique()
    G.add_nodes_from(nodes)

    for row in edge_w.itertuples(index=False):
        G.add_edge(row.u, row.v, weight=int(row.weight))

    communities = louvain_communities(G, weight="weight", seed=42)

    node2comm = {}
    for cid, comm_nodes in enumerate(communities):
        for n in comm_nodes:
            node2comm[n] = cid

    df["source_commu"] = df["source"].map(node2comm).astype("Int64")
    df["destination_commu"] = df["destination"].map(node2comm).astype("Int64")

    file_name = file_path.stem
    out_dir = Path("results") / file_name
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "louvain.csv"
    df.to_csv(out_path, index=False)

    return str(out_path)

In [3]:
out_path = louvain("syn_data/p0.8_mu0.2_l20.csv")

In [2]:
from pathlib import Path

input_dir = Path("syn_data")
csv_files = sorted(input_dir.glob("*.csv"))

print(f"Found {len(csv_files)} files.")

for file_path in csv_files:
    print(f"Processing: {file_path}")
    out_path = louvain(str(file_path))
    print(f"Saved to: {out_path}")

Found 100 files.
Processing: syn_data/p0.85_mu0.05_1.csv
Saved to: results/p0.85_mu0.05_1/louvain.csv
Processing: syn_data/p0.85_mu0.05_2.csv
Saved to: results/p0.85_mu0.05_2/louvain.csv
Processing: syn_data/p0.85_mu0.05_3.csv
Saved to: results/p0.85_mu0.05_3/louvain.csv
Processing: syn_data/p0.85_mu0.05_4.csv
Saved to: results/p0.85_mu0.05_4/louvain.csv
Processing: syn_data/p0.85_mu0.0_1.csv
Saved to: results/p0.85_mu0.0_1/louvain.csv
Processing: syn_data/p0.85_mu0.0_2.csv
Saved to: results/p0.85_mu0.0_2/louvain.csv
Processing: syn_data/p0.85_mu0.0_3.csv
Saved to: results/p0.85_mu0.0_3/louvain.csv
Processing: syn_data/p0.85_mu0.0_4.csv
Saved to: results/p0.85_mu0.0_4/louvain.csv
Processing: syn_data/p0.85_mu0.15_1.csv
Saved to: results/p0.85_mu0.15_1/louvain.csv
Processing: syn_data/p0.85_mu0.15_2.csv
Saved to: results/p0.85_mu0.15_2/louvain.csv
Processing: syn_data/p0.85_mu0.15_3.csv
Saved to: results/p0.85_mu0.15_3/louvain.csv
Processing: syn_data/p0.85_mu0.15_4.csv
Saved to: result